First, let’s go over the basics of GDSFactory and layout design.
The key to success is sticking to a hierarchical design flow and keeping everything as flexible as possible.

![My Image](Pic1.png)

In [ ]:
import gdsfactory as gf
import numpy as np
gf.gpdk.PDK.activate()
# Create a rectangular polygon to serve as our waveguide, where we'll define the input and output ports
# The function will return a GDS component with ports, allowing us to concatenate components

@gf.cell
def straight(length:float = 10, width:float = 1, layer:str=(1, 0),):
    c = gf.Component()
    c.add_polygon([(0, 0), (length, 0), (length, width), (0, width)], layer=layer)
    c.add_port(name="o1", center=(0, width / 2), width=width, orientation=180, layer=layer)
    c.add_port(name="o2", center=(length, width / 2), width=width, orientation=0, layer=layer)
    c.info["length"] = length
    c.info["width"] = width 
    return c

'''
First, we define the components, and then we can move or concatenate them. There are plenty of ways to do this—some more efficient than others.
GDSFactory has default functions for basic components as well.
'''
c = gf.Component()

wg1 = c << straight(length=6, width=0.5, layer=(1, 0))
wg2 = c << straight(length=10, width=2, layer=(1, 0))
bend = c << gf.components.bend_circular(radius=10, width=0.5, angle=90, layer=(1, 0))

wg1.movey(10)
wg2.drotate(90)
bend.connect(port="o1", other = wg2.ports["o1"], allow_width_mismatch=True)

c.plot()
print(wg1.cell.info['length'])

ModuleNotFoundError: No module named 'gdsfactory'

Now we’ll practice building complex structures from simpler ones, while always keeping access to the connection ports of each structure.

---------------------------------------      EXCERCISE 1       ---------------------------------------
---------------------------------------------------------------------------------------------------------------------

Try creating a function that generates the first letter of your name on a specified layer (MINUS). Make sure the width, size, and connection ports are flexible and adjustable


In [ ]:
##EXAMPLE##
###########
def p(L = 10, w = 1, layer=(1, 0)):
    c = gf.Component()
    ''' Create elements '''
    S1 = c << straight(L, w, layer)
    S2 = c << straight(2.5*L, w, layer)
    S3 = c << straight(2*L, w, layer)
    B1 = c << gf.components.bend_circular(radius=5, width=w, angle=180, layer=layer)
    B2 = c << gf.components.bend_circular(radius=5, width=w, angle=-45, layer=layer)
    ''' Move and connect elements '''
    S1.drotate(60)
    S2.drotate(-90)
    S2.move([L*np.cos(60*np.pi/180),L*np.sin(60*np.pi/180)])
    S3.connect(port="o1", other = S2.ports["o2"], allow_width_mismatch=True, allow_layer_mismatch = True, mirror = True)
    S3.mirror_y(S3.ports['o1'].center[1])
    B1.connect(port="o1", other = S3.ports["o2"], allow_width_mismatch=True, allow_layer_mismatch = True, mirror = True)
    B2.connect(port="o1", other = B1.ports["o2"], allow_width_mismatch=True, allow_layer_mismatch = True, mirror = True)
    ''' Add and inherit ports '''
    c.add_port("o1", port=S1.ports["o1"],port_type="optical")
    c.add_port("o2", port=B2.ports["o2"],port_type="optical")
    return c

c = gf.Component()

P = c << p(10, 0.5, layer=(1, 0))
P1 = c << p(10, 0.5, layer=(2, 0))

P1.connect(port="o1", other = P.ports["o2"], allow_width_mismatch=True, allow_layer_mismatch = True, mirror = False)
P1.drotate(105,center = [P.ports['o2'].center[0]/1000,P.ports['o2'].center[1]/1000])
c.add_ports(P.ports)
c.draw_ports()
c.plot()

In [ ]:
@gf.cell
def m_letter(L=10, w=1, layer=(1, 0)):
    c = gf.Component()

    # Segmentos verticales
    S_left = c << straight(length=L, width=w, layer=layer)
    S_right = c << straight(length=L, width=w, layer=layer)
    
    # Segmentos diagonales (usamos Pitágoras para que encajen bien)
    diag_length = L * 0.7  # Longitud aproximada para la diagonal
    D_left = c << straight(length=diag_length, width=w, layer=layer)
    D_right = c << straight(length=diag_length, width=w, layer=layer)

    # Rotamos la primera barra vertical para que esté de pie
    S_left.drotate(90)
    
    # Conectamos la primera diagonal al tope de la barra izquierda
    D_left.connect(port="o1", other=S_left.ports["o2"])
    D_left.drotate(-135, center=D_left.ports["o1"].center)
    
    # Conectamos la segunda diagonal a la primera
    D_right.connect(port="o1", other=D_left.ports["o2"])
    D_right.drotate(90, center=D_right.ports["o1"].center)
    
    # Conectamos la barra derecha al final de la segunda diagonal
    S_right.connect(port="o1", other=D_right.ports["o2"])
    S_right.drotate(-135, center=S_right.ports["o1"].center)
    
    # 3. Definir y heredar puertos
    c.add_port("o1", port=S_left.ports["o1"], port_type="optical")
    c.add_port("o2", port=S_right.ports["o2"], port_type="optical")
    
    return c


c = gf.Component("Ejercicio1_M")
m1 = c << m_letter(L=15, w=0.5, layer=(1, 0))
m2 = c << m_letter(L=15, w=0.5, layer=(2, 0))

# Conectamos una M con otra de diferente capa
m2.connect(port="o1", other=m1.ports["o2"])

c.plot()

---------------------------------------      EXCERCISE 2       ---------------------------------------
---------------------------------------------------------------------------------------------------------------------
Our next objective is to create a real MZI based on multimode interferometers and thermo-optic phase shifters. For this design, we will use the MMIs provided by the 'Foundry' and only need to design the PS section and assemble all the pieces together.

In [ ]:
##########################################################################################
##           This code section generates de MMI provided by the "Foundry" 
##########################################################################################
@gf.cell
def Taper(length_taper = 10, w_in = 1,  w_out = 1, layer=(1, 0)):
    ''' This code generates a taper '''
    c = gf.Component()
    c.add_polygon([(0, -w_in/2), (length_taper, -(w_in+w_out)/2), (length_taper, (w_in+w_out)/2), (0, w_in/2)], layer=layer)
    c.add_port(name="o1", center=[0, 0], width=w_in, orientation=180, layer=layer,port_type="optical")
    c.add_port(name="o2", center=[length_taper, 0], width=w_out, orientation=0, layer=layer,port_type="optical")
    c.info["length"] = length_taper
    return c
@gf.cell
def Core(l_core = 10, w_core = 1,layer=(1, 0)):
    ''' This code generates the core of the MMI '''
    c = gf.Component()
    c.add_polygon([(0, -w_core/2), (l_core, -w_core/2), (l_core, w_core/2), (0,w_core/2)], layer=layer)
    c.add_port(name="o1", center=[0, 0], width=w_core, orientation=180, layer=layer,port_type="optical")
    c.add_port(name="o2", center=[l_core, 0], width=w_core, orientation=0, layer=layer,port_type="optical")
    c.info["length"] = l_core
    return c
@gf.cell
def MMI(length_taper = 10, w_in = 1,  w_out = 1, sep = 5, l_core = 20, w_core = 10):
    ''' This code generates the complete MMI using a Core and Tapers '''
    c = gf.Component()
    Taper1 = c << Taper(length_taper, w_in,  w_out, layer=(1, 0))
    Taper2 = c << Taper(length_taper, w_in,  w_out, layer=(1, 0))
    Taper2.dmove((0, -sep)) 
    Core_MMI = c << Core(l_core, w_core,layer=(1, 0))
    Core_MMI.dmove((length_taper, -sep/2)) 

    Taper3 = c << Taper(length_taper, w_in,  w_out, layer=(1, 0))
    Taper4 = c << Taper(length_taper, w_in,  w_out, layer=(1, 0))
    
    Taper3.dmirror_x()
    Taper4.dmirror_x()
    Taper3.dmove((l_core + 2*length_taper , 0))
    Taper4.dmove((l_core + 2*length_taper , -sep))

    c.add_port("o1", port=Taper1.ports["o1"],port_type="optical") 
    c.add_port("o2", port=Taper2.ports["o1"],port_type="optical") 
    c.add_port("o3", port=Taper3.ports["o1"], orientation=180,port_type="optical") 
    c.add_port("o4", port=Taper4.ports["o1"], orientation=180,port_type="optical")
    c.info["length"] = l_core + 2*length_taper
    return c

##########################################################################################
##########################################################################################

c = gf.Component()
mmi = c << MMI(length_taper = 3, w_in = 0.5,  w_out = 0.5, sep = 1.5, l_core = 32, w_core = 5)
c.plot()

First, create the waveguide section with a heater above it. Remember that the heater must be electrically connected at the edges to drive it, and these connections should remain accessible for other functions.

Crossection table:
- Optical waveguide layer: Layer 1
- Electrical/metal layer: Layer 2 
- Heater layer: Layer 3

![My Image](Pic2.png)

In [ ]:
@gf.cell
def heater_phase_shifter(length=100, width=0.5, heater_width=2, pad_size=10):
    c = gf.Component()
    
    # 1. Guía de onda (Layer 1)
    wg = c << straight(length=length, width=width, layer=(1, 0))
    
    # 2. Calentador (Layer 3) - Lo centramos sobre la guía
    heater = c << straight(length=length, width=heater_width, layer=(3, 0))
    heater.dmovey(width/2 - heater_width/2)
    
    # 3. Contactos Metálicos (Layer 2) - En los extremos para poder soldar/conectar
    pad1 = c << straight(length=pad_size, width=pad_size, layer=(2, 0))
    pad2 = c << straight(length=pad_size, width=pad_size, layer=(2, 0))
    
    # Posicionamos los pads en los extremos del calentador
    pad1.dmove((0, width/2 - pad_size/2))
    pad2.dmove((length - pad_size, width/2 - pad_size/2))
    
    # Definimos los puertos ópticos (heredados de la guía)
    c.add_port("o1", port=wg.ports["o1"])
    c.add_port("o2", port=wg.ports["o2"])
    
    # Definimos los puertos eléctricos (en el centro de los pads)
    c.add_port("e1", center=pad1.dcenter, width=pad_size, orientation=90, layer=(2, 0), port_type="electrical")
    c.add_port("e2", center=pad2.dcenter, width=pad_size, orientation=90, layer=(2, 0), port_type="electrical")
    
    return c

In [ ]:
@gf.cell
def MZI_complete(ps_length=150, arm_separation=1.5):
    c = gf.Component()
    
    # Instanciamos los dos MMI
    mmi_in = c << MMI(length_taper=3, w_in=0.5, w_out=0.5, sep=arm_separation, l_core=32, w_core=5)
    mmi_out = c << MMI(length_taper=3, w_in=0.5, w_out=0.5, sep=arm_separation, l_core=32, w_core=5)
    
    # Instanciamos el Phase Shifter para el brazo superior
    ps = c << heater_phase_shifter(length=ps_length)
    
    # Instanciamos una guía recta para el brazo inferior (referencia)
    wg_bot = c << straight(length=ps_length, width=0.5, layer=(1, 0))
    
    # --- Conexiones ---
    # Nota: Usamos .connect() para que GDSFactory alinee los puertos automáticamente
    # Brazo Superior: MMI_in(o3) -> PS(o1) -> MMI_out(o1)
    ps.connect("o1", mmi_in.ports["o3"])
    mmi_out.connect("o1", ps.ports["o2"])
    
    # Brazo Inferior: MMI_in(o4) -> WG_bot(o1) -> MMI_out(o2)
    wg_bot.connect("o1", mmi_in.ports["o4"])
    
    # Puertos externos del MZI
    c.add_port("o1", port=mmi_in.ports["o1"])
    c.add_port("o2", port=mmi_in.ports["o2"])
    c.add_port("o3", port=mmi_out.ports["o3"])
    c.add_port("o4", port=mmi_out.ports["o4"])
    
    # Puertos eléctricos del calentador expuestos al exterior
    c.add_port("e1", port=ps.ports["e1"])
    c.add_port("e2", port=ps.ports["e2"])
    
    return c

# Visualización final
mzi_final = MZI_complete()
mzi_final.plot()

---------------------------------------      EXCERCISE 3       ---------------------------------------
---------------------------------------------------------------------------------------------------------------------
Create a tunale ring resonator using the elements from previous exercises (MZI, Heater...)